In [ ]:
import os
import io
import json
import logging
import asyncio
import httpx
from pydantic import BaseModel, Field
from openai import AsyncOpenAI

# -------------------------------------------------------------------
# 1. CONFIGURACIÓN DE LOGGING (DEBUG)
# -------------------------------------------------------------------
# Configuramos el formato para ver timestamps y niveles de log de forma clara
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler()]
)
logger = logging.getLogger("PerfumeAgentDebug")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")




In [21]:
# -------------------------------------------------------------------
# 2. DEFINICIÓN DE PROMPTS Y SCHEMAS (Pydantic)
# -------------------------------------------------------------------
SYSTEM_PERFUME_GUARDRAIL = """
### ROL Y PROPÓSITO
Eres "Aura", una Sommelier Olfativa Senior y Consultora Experta en Perfumería de Alta Gama, Nicho y Diseñador. Tu misión es guiar al usuario a encontrar su firma olfativa ideal mediante un análisis sensorial, técnico y estilístico.

### USO DE HERRAMIENTAS DE BÚSQUEDA WEB
Tienes acceso a la herramienta `web_search`.
- Si el usuario pregunta sobre lanzamientos recientes, precios en tiempo real, noticias o perfumes muy nuevos, DEBES invocar la función `web_search`.
- REGLA DE FIDELIDAD DE BÚSQUEDA (ESTRICTA): Cuando utilices `web_search`, basa tus afirmaciones sobre años de lanzamiento, nombres de perfumes y notas EXCLUSIVAMENTE en el texto retornado por la herramienta. NO inventes ni asumas que un perfume clásico o antiguo fue lanzado en el año actual solo porque el usuario lo mencionó en su pregunta. Si la búsqueda no revela ningún lanzamiento oficial para el año solicitado, acláralo amablemente como Sommelier.

### REGLAS DE ORO & GUARDRAILS (ESTRICTO)
1. TONO: Sofisticado, poético pero accesible, evocador y libre de pretensiones.
2. DELIMITACIÓN DE DOMINIO: Responderás EXCLUSIVAMENTE sobre perfumería, familias olfativas, notas y temas afines.
3. RECOMENDACIONES: Máximo 3 fragancias por intervención.
"""

WEB_SEARCH_TOOL = {
    "type": "function",
    "function": {
        "name": "web_search",
        "description": "Busca en la web información actualizada sobre lanzamientos de perfumes, notas olfativas, reseñas recientes, casas de perfumería o precios en tiempo real.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "La consulta de búsqueda optimizada para encontrar información de perfumería en la web (ej: 'Jo Milano Game of Spades latest release notes')."
                }
            },
            "required": ["query"]
        }
    }
}

class ChatMessage(BaseModel):
    role: str = Field(description="Rol del emisor: 'user', 'assistant' o 'system'")
    content: str = Field(description="Contenido del mensaje")

class PerfumeAgentResponse(BaseModel):
    transcription: str | None = Field(default=None, description="Transcripción devuelta por Whisper")
    response: str = Field(description="Respuesta generada por Aura")
    updated_history: list[ChatMessage] = Field(description="Historial actualizado")
    tool_was_called: bool = Field(default=False, description="Indicador de si la herramienta fue invocada")



In [24]:
# -------------------------------------------------------------------
# 3. FUNCIONES DEL AGENTE CON TRAZABILIDAD Y LOGS
# -------------------------------------------------------------------
async def perform_web_search(query: str) -> str:
    """
    Realiza búsquedas web asegurando un contexto informativo rico para el LLM.
    """
    try:
        tavily_key = "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"
        if tavily_key:
            async with httpx.AsyncClient() as http_client:
                res = await http_client.post(
                    "https://api.tavily.com/search",
                    json={
                        "api_key": tavily_key, 
                        "query": query, 
                        "max_results": 5,
                        "search_depth": "advanced"
                    }
                )
                data = res.json()
                results = [
                    f"Fuente: {r.get('title')}\nContenido: {r.get('content')}" 
                    for r in data.get("results", [])
                ]
                if results:
                    return "INFORMACIÓN EN TIEMPO REAL OBTENIDA DE LA WEB:\n" + "\n---\n".join(results)
                return "No se encontraron resultados relevantes en la web para esta consulta."

        # Fallback con DuckDuckGo API (JSON libre/público)
        async with httpx.AsyncClient() as http_client:
            res = await http_client.get(
                "https://api.duckduckgo.com/",
                params={"q": query, "format": "json", "no_html": 1},
                headers={"User-Agent": "Mozilla/5.0"}
            )
            data = res.json()
            abstract = data.get("AbstractText", "")
            related = [
                t.get("Text") for t in data.get("RelatedTopics", []) 
                if isinstance(t, dict) and "Text" in t
            ]
            
            combined = []
            if abstract:
                combined.append(f"Resumen: {abstract}")
            if related:
                combined.append("Detalles relacionados:\n" + "\n".join(related[:3]))

            if combined:
                return "INFORMACIÓN OBTENIDA DE LA WEB:\n" + "\n".join(combined)
            
            return f"Búsqueda realizada para '{query}'. No se hallaron artículos ni bases de datos actualizadas con un lanzamiento bajo ese criterio específico."

    except Exception as e:
        return f"Error técnico al ejecutar la búsqueda web: {str(e)}"


async def process_perfume_chat_debug(
    user_message: str,
    history: list[ChatMessage],
    transcription: str | None = None
) -> PerfumeAgentResponse:
    """Procesa la interacción con el agente detallando el uso de Function Calling."""
    
    api_key = OPENAI_API_KEY
    if not api_key:
        raise ValueError("🔑 OPENAI_API_KEY no encontrada en las variables de entorno.")

    client = AsyncOpenAI(api_key=api_key)

    # 1. Preparar mensajes
    messages = [{"role": "system", "content": SYSTEM_PERFUME_GUARDRAIL}]
    for msg in history:
        if msg.role in ["user", "assistant"]:
            messages.append({"role": msg.role, "content": msg.content})

    clean_input = user_message.replace("<", "").replace(">", "")
    messages.append({"role": "user", "content": clean_input})

    logger.info("=" * 60)
    logger.info(f"📩 [NUEVA CONSULTA] '{clean_input}'")
    logger.info(f"📊 Historial de conversación previo: {len(history)} mensajes")

    tool_called_flag = False

    try:
        # 2. Primera llamada a gpt-4o-mini
        logger.info("🤖 [LLM STEP 1] Evaluando mensaje con gpt-4o-mini...")
        
        response = await client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=[WEB_SEARCH_TOOL],
            tool_choice="auto",
            temperature=0.5,
            timeout=25.0
        )

        response_message = response.choices[0].message

        # 3. Inspeccionar decisión del modelo
        if response_message.tool_calls:
            tool_called_flag = True
            logger.info("⚙️ [DECISIÓN] El modelo DETERMINÓ que necesita usar 'web_search'.")
            messages.append(response_message)

            for tool_call in response_message.tool_calls:
                logger.info(f"   ↳ ID Llamada: {tool_call.id}")
                logger.info(f"   ↳ Función: {tool_call.function.name}")
                logger.info(f"   ↳ Argumentos RAW: {tool_call.function.arguments}")

                if tool_call.function.name == "web_search":
                    args = json.loads(tool_call.function.arguments)
                    search_query = args.get("query", clean_input)

                    # Ejecutar búsqueda
                    search_result = await perform_web_search(search_query)

                    # Inyectar respuesta del tool
                    messages.append({
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "name": "web_search",
                        "content": search_result
                    })

            # 4. Segunda llamada para consolidar respuesta
            logger.info("🤖 [LLM STEP 2] Generando respuesta final basada en datos de la búsqueda...")
            second_response = await client.chat.completions.create(
                model="gpt-4o-mini",
                messages=messages,
                temperature=0.6
            )
            agent_reply = second_response.choices[0].message.content

        else:
            logger.info("🚫 [DECISIÓN] El modelo DECIDIÓ NO USAR la herramienta web_search. Responderá directo.")
            agent_reply = response_message.content

        # 5. Generar historial actualizado
        updated_history = [
            *history,
            ChatMessage(role="user", content=clean_input),
            ChatMessage(role="assistant", content=agent_reply)
        ]

        return PerfumeAgentResponse(
            transcription=transcription,
            response=agent_reply,
            updated_history=updated_history,
            tool_was_called=tool_called_flag
        )

    except Exception as e:
        logger.error(f"❌ [ERROR GENERAL] {str(e)}")
        raise e

In [25]:
# -------------------------------------------------------------------
# 4. BUCLE DE PRUEBAS INTERACTIVO EN JUPYTER
# -------------------------------------------------------------------

# Ingrese su API Key si no está configurada en las variables de entorno del sistema


async def ejecutar_pruebas():
    history = []

    print("\n--- PRUEBA 1: Pregunta general (NO debería activar la Tool) ---")
    res1 = await process_perfume_chat_debug(
        user_message="¿Qué notas caracterizan a un perfume Oriental o Ambarado?",
        history=history
    )
    history = res1.updated_history
    print(f"\n🔮 [Respuesta de Aura]:\n{res1.response}\n")

    print("\n--- PRUEBA 2: Consulta sobre lanzamiento reciente (SÍ debería activar la Tool) ---")
    res2 = await process_perfume_chat_debug(
        user_message="¿Cuáles son las notas olfativas y el precio del último perfume masculino lanzado por Creed en 2026?",
        history=history
    )
    history = res2.updated_history
    print(f"\n🔮 [Respuesta de Aura]:\n{res2.response}\n")

# Ejecución en entornos de Jupyter (asynchio friendly)
await ejecutar_pruebas()

2026-07-27 11:52:32,646 [INFO] ============================================================
2026-07-27 11:52:32,647 [INFO] 📩 [NUEVA CONSULTA] '¿Qué notas caracterizan a un perfume Oriental o Ambarado?'
2026-07-27 11:52:32,647 [INFO] 📊 Historial de conversación previo: 0 mensajes
2026-07-27 11:52:32,648 [INFO] 🤖 [LLM STEP 1] Evaluando mensaje con gpt-4o-mini...



--- PRUEBA 1: Pregunta general (NO debería activar la Tool) ---


2026-07-27 11:52:37,446 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-07-27 11:52:37,459 [INFO] 🚫 [DECISIÓN] El modelo DECIDIÓ NO USAR la herramienta web_search. Responderá directo.
2026-07-27 11:52:37,516 [INFO] ============================================================
2026-07-27 11:52:37,517 [INFO] 📩 [NUEVA CONSULTA] '¿Cuáles son las notas olfativas y el precio del último perfume masculino lanzado por Creed en 2026?'
2026-07-27 11:52:37,517 [INFO] 📊 Historial de conversación previo: 2 mensajes
2026-07-27 11:52:37,517 [INFO] 🤖 [LLM STEP 1] Evaluando mensaje con gpt-4o-mini...



🔮 [Respuesta de Aura]:
Los perfumes de la familia Oriental, también conocidos como Ambarados, son una celebración de la calidez y la sensualidad. Estas fragancias se caracterizan por una rica y compleja combinación de notas que evocan un ambiente exótico y envolvente. 

### Notas Características:

1. **Notas de Fondo**:
   - **Ámbar**: Un componente esencial que aporta calidez y dulzura.
   - **Resinas**: Como el incienso y la mirra, que añaden profundidad y un toque místico.
   - **Maderas**: La madera de sándalo y el cedro son comunes, brindando una base terrosa y reconfortante.

2. **Notas de Corazón**:
   - **Especias**: Canela, clavo y pimienta son frecuentemente utilizadas, aportando un carácter picante y cálido.
   - **Flores**: A menudo se combinan con flores como el jazmín o la rosa, que añaden un toque de elegancia y feminidad.

3. **Notas de Salida**:
   - **Frutas**: Algunas fragancias orientales pueden comenzar con notas frutales, como la mandarina o la pera, que ofrecen 

2026-07-27 11:52:38,548 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-07-27 11:52:38,551 [INFO] ⚙️ [DECISIÓN] El modelo DETERMINÓ que necesita usar 'web_search'.
2026-07-27 11:52:38,551 [INFO]    ↳ ID Llamada: call_1V5GDKVFPitIBUvuwA8QXSri
2026-07-27 11:52:38,552 [INFO]    ↳ Función: web_search
2026-07-27 11:52:38,553 [INFO]    ↳ Argumentos RAW: {"query":"Creed latest men's perfume release 2026 notes and price"}
2026-07-27 11:52:41,812 [INFO] HTTP Request: POST https://api.tavily.com/search "HTTP/1.1 200 OK"
2026-07-27 11:52:41,816 [INFO] 🤖 [LLM STEP 2] Generando respuesta final basada en datos de la búsqueda...
2026-07-27 11:52:44,093 [INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"



🔮 [Respuesta de Aura]:
El último perfume masculino lanzado por Creed en 2026 es el **Wild Vetiver**. Aunque los detalles específicos sobre sus notas olfativas no se han revelado completamente, se espera que contenga un enfoque fresco y limpio, similar a su predecesor, Original Vetiver, pero con un matiz más destacado de vetiver.

### Precio
El precio de **Wild Vetiver** es de aproximadamente **320 euros**.

Este perfume promete ser una adición intrigante a la oferta de Creed, especialmente para aquellos que aprecian la elegancia y la frescura de las fragancias con vetiver.



In [ ]:
(async () => {
  console.log("🔍 --- DIAGNÓSTICO PERFIL IA & RECOMENDACIONES (FASTAPI) ---");

  // 1. Intentar obtener el token de autenticación
  const token = localStorage.getItem('token') || 
                localStorage.getItem('auth_token') || 
                sessionStorage.getItem('token') || 
                sessionStorage.getItem('auth_token');

  console.log("🔑 Token:", token ? "Presente (OK)" : "⚠️ No encontrado en storage local");

  // 2. Intentar decodificar el userId desde el JWT o localStorage
  let userId = null;
  if (token) {
    try {
      const payloadBase64 = token.split('.')[1];
      const decodedPayload = JSON.parse(atob(payloadBase64));
      userId = decodedPayload.sub || decodedPayload.user_id || decodedPayload.id;
      console.log("👤 User ID extraído del Token JWT:", userId);
    } catch (e) {
      console.warn("⚠️ No se pudo decodificar el token JWT directamente.");
    }
  }

  // Fallback si no está en el token: intentar buscarlo en objetos guardados en localStorage
  if (!userId) {
    const rawUser = localStorage.getItem('user') || localStorage.getItem('currentUser');
    if (rawUser) {
      try {
        const parsed = JSON.parse(rawUser);
        userId = parsed.id || parsed.user_id || parsed._id;
        console.log("👤 User ID extraído de localStorage:", userId);
      } catch(e) {}
    }
  }

  // SI NO SE ENCUENTRA USER ID AUTOMÁTICAMENTE, REEMPLÁZALO AQUÍ MANUALLMENTE:
  if (!userId) {
    userId = "628be202-b13e-4cac-b94e-6ab4c62a0ba1"; // <-- pon tu ID real si sale undefined
    console.log("⚠️ Usando userId manual:", userId);
  }

  // 3. Define la URL base de tu backend FastAPI (ej. Render o tu dominio de API)
  // Reemplaza 'https://tu-backend-en-render.onrender.com' por la URL real de tu API si es cross-origin
  const API_BASE = "https://scentia-prod-v1.onrender.com"; // <-- AJUSTA SEGÚN TU BACKEND REAL
  
  // Ajusta el prefijo del router si usas '/api' o '/api/v1' antes de {user_id}
  const endpoint = `${API_BASE}/api/users/${userId}/ai-profile`;

  try {
    console.log(`📡 Enviando GET a: ${endpoint}...`);

    const headers = { 'Content-Type': 'application/json' };
    if (token) headers['Authorization'] = `Bearer ${token}`;

    const res = await fetch(endpoint, { method: 'GET', headers });

    console.log(`STATUS HTTP: ${res.status} ${res.statusText}`);

    if (!res.ok) {
      const errorDetail = await res.text();
      console.error("❌ El backend respondió con un error:", errorDetail);
      return;
    }

    const data = await res.json();
    console.log("📦 PAYLOAD RECIBIDO DEL BACKEND:", data);

    console.log("\n📊 --- REVISIÓN DE LAS RECOMENDACIONES ---");
    console.log("• ¿'recommendations' está en el JSON?:", 'recommendations' in data);
    console.log("• Tipo:", typeof data.recommendations);
    console.log("• Cantidad de elementos (length):", data.recommendations?.length);

    if (Array.isArray(data.recommendations)) {
      if (data.recommendations.length > 0) {
        console.log("🎉 ¡Las recomendaciones SÍ están llegando desde el backend! Primer elemento:", data.recommendations[0]);
      } else {
        console.warn("⚠️ 'recommendations' llegó como un array VACÍO [].");
        console.warn("Motivos posibles: 'ml_analysis' devolvió None en el backend o 'generate_recommendations' no encontró ítems.");
      }
    }

  } catch (err) {
    console.error("💥 Error de conexión/CORS con el backend:", err);
  }
})();